In [1]:
from datasets import load_dataset

dataset_dict = load_dataset("SetFit/ag_news", split='train', download_mode="reuse_dataset_if_exists")
remove_columns = dataset_dict.column_names
raw_labels = dataset_dict.unique('label')
label2id = {str(label): idx for idx, label in enumerate(sorted(map(str, raw_labels)))}
id2label = {idx: label for label, idx in label2id.items()}

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /datasets/SetFit/ag_news/resolve/main/README.md (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))"), '(Request ID: 3abbf3a6-6afa-4385-b84b-a18d00b4bc70)')' thrown while requesting HEAD https://huggingface.co/datasets/SetFit/ag_news/resolve/main/README.md
Retrying in 1s [Retry 1/5].


In [2]:
from copy import deepcopy

from transformers import AutoModelForSequenceClassification, AutoTokenizer

from shadow_peft import ShadowForSequenceClassification

shadow_peft_path = "erin99/agnews_shadow_peft"
tokenizer = AutoTokenizer.from_pretrained(shadow_peft_path)

base_model = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen3-0.6B",
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    ).to('cuda:7')

model = ShadowForSequenceClassification.from_pretrained(
    deepcopy(base_model), shadow_peft_path, is_trainable=False, inference_mode='base_shadow').to('cuda:7')
shadow_only = ShadowForSequenceClassification.from_pretrained(
    deepcopy(base_model), shadow_peft_path, is_trainable=False, inference_mode='shadow_only').to('cuda:7')

model = model.eval()
shadow_only = shadow_only.eval()


Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 27176.92it/s]


In [3]:
import time

import torch

In [4]:
doc = '''
Owners Seek Best Ballpark Deal for Expos (AP) AP - Trying to get the best possible ballpark deal for the Montreal Expos, major league baseball instructed its lawyers to press ahead with negotiations involving four of the areas bidding for the team.
'''

tok = tokenizer(doc, return_tensors='pt')
tok = tok.to(model.device)
s_time = time.time()
with torch.no_grad():
    outputs = model(**tok)
print('Inference time:', time.time() - s_time)
prediction = outputs.logits.argmax().item()
shadow_prediction = outputs.shadow_logits.argmax().item()

id2label[prediction], id2label[shadow_prediction]

Inference time: 0.6193866729736328


('1', '1')

In [8]:
s_time = time.time()
with torch.no_grad():
    outputs = shadow_only(**tok)
print('Inference time:', time.time() - s_time)
prediction = outputs.logits.argmax().item()
shadow_prediction = outputs.shadow_logits.argmax().item()

id2label[shadow_prediction]

Inference time: 0.00467371940612793


'1'

In [11]:
model.push_to_hub('erin99/agnews_shadow_peft_test', private=True)

Processing Files (2 / 2): 100%|██████████| 36.3MB / 36.3MB, 7.27MB/s  
New Data Upload: 100%|██████████| 36.3MB / 36.3MB, 7.27MB/s  


'https://huggingface.co/erin99/agnews_shadow_peft_test/commit/00a247d9a82acb3a0173d681204ae0705273d290'